# Customer Conversion Optimization & Prediction  
## 2. Feature Engineering

This notebook prepares the dataset for modeling by:
*  ⁠Creating the binary target
*  ⁠Engineering new predictive features
*  ⁠Encoding categorical variables
*  ⁠Exporting the processed dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np

### Load the Dataset

In [2]:
# Load raw dataset
df = pd.read_csv("../data/bank-full.csv", sep=";")

# Display shape and preview
print("Raw shape:", df.shape)
df.head()

Raw shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


### Create Binary Target Variable

In [3]:
# Convert target column to binary
df["target"] = df["y"].map({"yes": 1, "no": 0})

# Verify target mapping
df[["y", "target"]].head()

,y,target
0,no,0
1,no,0
2,no,0
3,no,0
4,no,0


### Feature Engineering
The goal here is to create additional variables that may better capture customer and campaign behavior.

In [4]:
# Log-transform balance to reduce skewness
# Negative values are clipped at 0 before log1p
df["balance_log"] = np.log1p(df["balance"].clip(lower=0))

# Flag whether the customer was contacted previously
df["was_contacted_before"] = (df["previous"] > 0).astype(int)

# Flag whether the customer had recent previous contact
# pdays = number of days since last contact; -1 means never contacted
df["recent_contact"] = ((df["pdays"] > -1) & (df["pdays"] <= 30)).astype(int)

# Group number of campaign contacts into interpretable bands
df["campaign_group"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 3, 5, 100],
    labels=["1", "2-3", "4-5", "6+"],
    include_lowest=True
)

# Create customer age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 65, 120],
    labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"],
    include_lowest=True
)

# Financial pressure score:
# Higher score may reflect more financial obligations
df["financial_pressure_score"] = (
    (df["housing"] == "yes").astype(int)
    + (df["loan"] == "yes").astype(int)
    + (df["default"] == "yes").astype(int)
)

# Overall contact intensity
df["contact_intensity"] = df["campaign"] + df["previous"]

### Review Engineered Features

In [5]:
# Display selected engineered columns
engineered_cols = [
    "balance_log",
    "was_contacted_before",
    "recent_contact",
    "campaign_group",
    "age_group",
    "financial_pressure_score",
    "contact_intensity"
]

df[engineered_cols].head()

,balance_log,was_contacted_before,recent_contact,campaign_group,age_group,financial_pressure_score,contact_intensity
0,7.670429,0,0,1,56-65,1,1
1,3.401197,0,0,1,36-45,1,1
2,1.098612,0,0,1,26-35,2,1
3,7.317876,0,0,1,46-55,1,1
4,0.693147,0,0,1,26-35,0,1


### Drop Original Non-Binary Target
The original 'y' column is removed because 'target' is the model-ready version.

In [8]:
# Drop original target label
df = df.drop(columns=["y"])

### Encode Categorical Variables
Machine learning models require numeric inputs, so categorical variables are one-hot encoded.

In [9]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Check processed shape
print("Processed shape:", df_encoded.shape)

# Preview processed dataset
df_encoded.head()

Processed shape: (45211, 56)


,age,balance,day,duration,campaign,pdays,previous,target,balance_log,was_contacted_before,...,poutcome_success,poutcome_unknown,campaign_group_2-3,campaign_group_4-5,campaign_group_6+,age_group_26-35,age_group_36-45,age_group_46-55,age_group_56-65,age_group_65+
0,58,2143,5,261,1,-1,0,0,7.670429,0,...,False,True,False,False,False,False,False,False,True,False
1,44,29,5,151,1,-1,0,0,3.401197,0,...,False,True,False,False,False,False,True,False,False,False
2,33,2,5,76,1,-1,0,0,1.098612,0,...,False,True,False,False,False,True,False,False,False,False
3,47,1506,5,92,1,-1,0,0,7.317876,0,...,False,True,False,False,False,False,False,True,False,False
4,33,1,5,198,1,-1,0,0,0.693147,0,...,False,True,False,False,False,True,False,False,False,False


### Save Processed Data
The processed file is saved to the 'outputs' folder for use in modeling.

In [11]:
# Save processed dataset
df_encoded.to_csv("../outputs/processed_data.csv", index=False)

print("Processed dataset saved to ../outputs/processed_data.csv")

Processed dataset saved to ../outputs/processed_data.csv


### Quick Feature Checks
These summaries help confirm that the engineered features are meaningful.

In [17]:
# Conversion by campaign group
df.groupby("campaign_group")["target"].mean()

/var/folders/yh/ly0nbbjs68z4r20sq1qh_6nm0000gn/T/ipykernel_78067/2987170531.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("campaign_group")["target"].mean()


campaign_group
1      0.145976
2-3    0.112005
4-5    0.086266
6+     0.058094
Name: target, dtype: float64

In [16]:
# Conversion by age group
df.groupby("age_group")["target"].mean()

/var/folders/yh/ly0nbbjs68z4r20sq1qh_6nm0000gn/T/ipykernel_78067/1199415478.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("age_group")["target"].mean()


age_group
<=25     0.239521
26-35    0.120031
36-45    0.093894
46-55    0.093527
56-65    0.141239
65+      0.426099
Name: target, dtype: float64

In [18]:
# Conversion by financial pressure score
df.groupby("financial_pressure_score")["target"].mean()

financial_pressure_score
0    0.183616
1    0.080147
2    0.061224
3    0.052632
Name: target, dtype: float64

### Conclusion
The dataset is now:
* ⁠cleaned
* feature-engineered
* ⁠encoded
* ⁠ready for modeling

The next notebook focuses on model training, evaluation, and business interpretation.